# 01 · PDF → Markdown 流水线（DeepSeek-OCR-2）

**硬件**：🟡 需要 NVIDIA GPU 16GB+（依赖 flash-attn，Mac 无法本地跑——可用云 GPU，或跳到第 4 节的通用 VLM 备选方案）

## 本 notebook 你将学到

1. 文档流水线的标准骨架：**PDF → 页面图像 → OCR 模型 → Markdown 拼接**
2. DeepSeek-OCR 系列的核心参数：分辨率档位（`base_size`/`crop_mode`）如何权衡质量与 token 成本
3. grounded 输出（带坐标）为什么是防幻觉、可溯源的关键
4. 简易质量检查：怎么发现 OCR 结果里的"自信编造"

> 版本提示（2026-08）：环境要求以 [DeepSeek-OCR-2 模型卡](https://huggingface.co/deepseek-ai/DeepSeek-OCR-2) 为准（发布时要求 torch 2.6 + flash-attn 2.7.3 + transformers 4.46）。

In [ ]:
%pip install -q torch transformers tokenizers einops addict easydict pypdfium2 pillow
# GPU 环境还需要: %pip install -q flash-attn --no-build-isolation

## 1. PDF → 页面图像

OCR VLM 的输入是图像，第一步永远是渲染。**DPI 是第一个质量旋钮**：太低小字糊掉，太高浪费 token。150–200 DPI 是常见起点。

In [ ]:
import pypdfium2 as pdfium
import requests, pathlib

# 示例：拿一篇 arXiv 论文（Janus-Pro）当测试文档，换成你自己的 PDF 即可
pdf_path = pathlib.Path("sample.pdf")
if not pdf_path.exists():
    pdf_path.write_bytes(requests.get("https://arxiv.org/pdf/2501.17811", timeout=60).content)

pdf = pdfium.PdfDocument(pdf_path)
print(f"共 {len(pdf)} 页")

pages_dir = pathlib.Path("pages"); pages_dir.mkdir(exist_ok=True)
page_files = []
for i in range(min(3, len(pdf))):  # 演示只取前 3 页
    bitmap = pdf[i].render(scale=200 / 72)  # 200 DPI
    img = bitmap.to_pil()
    f = pages_dir / f"page_{i:03d}.png"
    img.save(f)
    page_files.append(f)
    print(f"{f}  {img.size}")

## 2. 加载 DeepSeek-OCR-2

模型走 `trust_remote_code`（推理逻辑在模型仓库里）。运行前读一眼它的 remote code 是好习惯。

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR-2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_safetensors=True,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
).eval().cuda()

## 3. 逐页 OCR → 拼接 Markdown

两个关键点：

- 提示词里的 **`<|grounding|>`** 让输出带版面坐标——每段文字能定位回原图，这是"可溯源"的基础
- **`base_size` / `crop_mode`** 控制视觉 token 预算：小档位便宜但小字/密表容易错，这正是 02 章理论讲的"光学压缩比"旋钮

In [ ]:
PROMPT = "<image>\n<|grounding|>Convert the document to markdown."

md_pages = []
for f in page_files:
    res = model.infer(
        tokenizer,
        prompt=PROMPT,
        image_file=str(f),
        output_path="ocr_out",
        base_size=1024, image_size=640, crop_mode=True,  # 档位参数见模型卡
        save_results=True,
    )
    md_pages.append(res if isinstance(res, str) else open(f"ocr_out/{f.stem}.md").read())
    print(f"{f.name} done")

full_md = "\n\n---\n\n".join(md_pages)
pathlib.Path("output.md").write_text(full_md)
print(full_md[:1500])

## 4. 备选方案：没有 NVIDIA GPU 时用通用 VLM

质量上限低一些、没有 grounding，但 Mac/API 都能跑。也正好可以和专用模型对比（这就是 notebooks/README 里 `02_vlm_vs_ocr_model` 的雏形）。

In [ ]:
# 用 01 章的 Qwen3-VL 或任一家闭源 API 逐页转写:
# prompt = "把这页文档完整转成 Markdown。表格用 Markdown 表格。读不清的字符用 � 标记，禁止猜测。"
# 复用 01-vlm/notebooks/01_qwen3vl_local.ipynb 的 chat() 即可，此处从略。

## 5. 质量检查：抓"自信编造"

端到端 OCR 最危险的失败模式不是漏字，而是**把读不清的内容编得很通顺**。两个便宜的体检手段：

1. **数字校验**：正则抽出所有数字，与原文抽样人工核对——数字是幻觉重灾区且后果最重
2. **回环校验**：把生成的 Markdown 渲染回图像，用另一个 VLM 对比两张图的差异（贵但全自动）

In [ ]:
import re
numbers = re.findall(r"\d+\.?\d*", full_md)
print(f"提取到 {len(numbers)} 个数字，抽样: {numbers[:20]}")
print("人工核对其中 10 个与原 PDF 是否一致——这是最便宜的验收测试。")

## 练习

1. 换 `base_size`（768/1024/1280）重跑同一页，对比公式和小字号脚注的识别质量——画出"质量 vs token 数"曲线。
2. 找一张倾斜拍摄的发票照片（而非扫描件），看流水线在真实世界输入下的表现。
3. 用 [OmniDocBench](https://github.com/opendatalab/OmniDocBench) 的样例子集给你的流水线打个分。
4. 把输出的 Markdown 切块灌进向量库——你已经完成了 08 章多模态 RAG 的前半段。